### Week 12 Review Question 2 - BERT word embedding in Python
<author> &copy; Prepared by Professor Yuefeng Li </author>


In [99]:
# Installing transformers and torch modules if your computer does not insatll them
# !pip install transformers
# !pip3 install torch torchvision

In [1]:
# importing libraries
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Set a random seed
random_seed = 42
random.seed(random_seed)
# Set a random seed for PyTorch (for GPU as well)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

In [3]:
# Load BERT tokenizer and model, where 
#'bert-base-uncased' is one of the standard pretrained BERT base models released by Google:
## text is lowercased, Hidden size 768, about 110M parameters
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

#### Week 3 Review Question 7

In [5]:
# Input texts
text1 = "Machinelearning for Natural Language Processing"
text2 = "Machinelearning for text data analysis"
# Tokenize and encode text using batch_encode_plus
# The function returns a dictionary containing the token IDs and attention masks
encoding = tokenizer.batch_encode_plus( [text1, text2],# List of input texts
    padding=True,              # Pad to the maximum sequence length
    truncation=True,           # Truncate to the maximum sequence length if necessary
    return_tensors='pt',      # Return PyTorch tensors
    add_special_tokens=True    # Add special tokens CLS and SEP
)

In [6]:
# Generate embeddings using BERT model
input_ids = encoding['input_ids']  # Token IDs
attention_mask = encoding['attention_mask']  # Attention mask
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)
    word_embeddings = outputs.last_hidden_state  # This contains the embeddings

print("Shape of Word Embeddings:") 
print(word_embeddings.shape)
#The shape of Word embeddings is [1, 12, 768] where 768 is the dimensionality/hidden size of the word embeddings, i.e.,  
# Each token is represented by a 768-dimensional vector and 10 is the number of tokens in our input text after tokenization. 
# Here 2 is the total number of sentences we have passed.

Shape of Word Embeddings:
torch.Size([2, 10, 768])


In [13]:
# Compute the average of word embeddings to get the sentence embedding
sentence_embedding = word_embeddings.mean(dim=1)  # Average pooling along the sequence length dimension

# Output the shape of the sentence embedding
print("Shape of Sentence Embedding:")
print(sentence_embedding.shape)

Shape of Sentence Embedding:
torch.Size([2, 768])


In [15]:
# Compute cosine similarity between text1 and text2
sent1_emb = sentence_embedding[0]
print(sent1_emb[:10])
sent2_emb = sentence_embedding[1]
similarity_score = cosine_similarity(sent1_emb.unsqueeze(0), sent2_emb.unsqueeze(0))
# Print the similarity score
print("Cosine Similarity Score:")
print(similarity_score[0][0])

tensor([-0.1828, -0.2940, -0.1903, -0.1386,  0.2304, -0.0658, -0.0335,  0.5041,
        -0.4737, -0.1669])
Cosine Similarity Score:
0.9200219


#### Week 12 Review Question 2 - Solution 
to better understand how the attention_mask is used to obtain encoder outputs

In [35]:
# (1) Tokenize an input text
inputs = tokenizer(text1, 
                   padding=True,
                   truncation=True, 
                   return_tensors="pt",
                   add_special_tokens=True)
# Get token embeddings layer from BERT 
embedding_layer = model.embeddings

In [37]:
# (2) Generate initial embeddings 
embeddings = embedding_layer( 
    input_ids=inputs["input_ids"], 
    token_type_ids=inputs["token_type_ids"] )

In [41]:
print("\nEmbedding Shape:") 
print(embeddings.shape)
print("\nFirst Token Embedding:") 
print(embeddings[0, 0, :20])


Embedding Shape:
torch.Size([1, 10, 768])

First Token Embedding:
tensor([ 0.1686, -0.2858, -0.3261, -0.1122,  0.0343, -0.2689, -0.0302, -0.0390,
         0.0157, -0.2828,  0.1436,  0.0426, -0.2203, -0.0023, -0.5525,  0.1092,
        -0.0211, -0.0151,  0.0724, -0.3034], grad_fn=<SliceBackward0>)


In [29]:
# Notes that: when we input text into BERT, all sequences must have the same length. 
# So shorter sentences are padded with special tokens (usually set padding=True in "tokenizer")

In [51]:
# (3) Pass Embeddings into BERT Encoder 

encoder_outputs = model.encoder( 
    embeddings, 
    attention_mask = inputs["attention_mask"].bool()
)

# the attention_mask tells the encoder: Which tokens are real words and which tokens are just padding.
# It is usually a binary mask: 1 - attend to this token, and 0 - ignore this token (usually padding)

In [49]:
# (4) print out the final contextual representations 
last_hidden_state = encoder_outputs.last_hidden_state 
print("\nEncoder Output Shape:") 
print(last_hidden_state.shape) 
# Example contextualized vector 
print("\nFirst Token Contextualized Representation:") 
print(last_hidden_state[0, 0, :20])


Encoder Output Shape:
torch.Size([1, 10, 768])

First Token Contextualized Representation:
tensor([-0.5284, -0.4093, -0.4727, -0.1305, -0.1049,  0.0786, -0.1079,  0.4094,
        -0.6169, -0.2246, -0.0838,  0.1648, -0.0798, -0.3042, -0.2275, -0.0727,
        -0.5986,  0.4513,  0.2165,  0.0188], grad_fn=<SliceBackward0>)
